# Student Depression Prediction
### SENG 352 Notebook-First Final Delivery

---

A complete ML pipeline exploring the drivers of student depression — from data quality to explainable predictions and interactive demo.

## Notebook Roadmap (11 Sections)
1. Problem, aim, and disclaimer
2. Data Quality Assessment (DQA)
3. Exploratory Data Analysis (EDA)
4. Preprocessing + Feature Engineering
5. Modeling + Hyperparameter Tuning
6. Test Evaluation + Error Analysis
7. Explainability (SHAP + LIME-ready)
8. Fairness + Statistical Significance + Confidence Interval
9. Model Card Summary + Ethics + Limitations
10. Gradio Interactive Demo
11. Final Results + Checklist Evidence Matrix

> This notebook is for educational analysis and early-warning discussion only. It is **not** a clinical diagnostic tool.

In [1]:
!pip install -q missingno shap plotly xgboost lightgbm imbalanced-learn gradio
print('Colab packages installed.')

Colab packages installed.


In [2]:
import os, pickle, time, warnings, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches
import seaborn as sns
import missingno as msno
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import pointbiserialr
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report,
    precision_recall_curve, roc_auc_score, roc_curve,
    f1_score, average_precision_score,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate,
    train_test_split, learning_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

try:
    from xgboost import XGBClassifier; HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier; HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    import shap; HAS_SHAP = True
except ImportError:
    HAS_SHAP = False; print('SHAP not available')

warnings.filterwarnings('ignore')
%matplotlib inline

# ── THEME ENGINE ──────────────────────────────────────────────────────────────
THEME = {
    'bg':        '#F5FBDA',
    'secondary': '#D9EFBD',
    'primary':   '#B9D175',
    'accent':    '#A33E79',
    'dark':      '#450C3F',
}
C_NO  = THEME['primary']
C_YES = THEME['accent']
PALETTE = {0: C_NO, 1: C_YES}
PALETTE_LIST = [C_NO, C_YES]

mpl.rcParams.update({
    'figure.facecolor':  THEME['bg'],
    'axes.facecolor':    THEME['bg'],
    'axes.edgecolor':    THEME['secondary'],
    'axes.linewidth':    0.8,
    'text.color':        THEME['dark'],
    'axes.labelcolor':   THEME['dark'],
    'xtick.color':       THEME['dark'],
    'ytick.color':       THEME['dark'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'axes.grid.axis':    'y',
    'grid.color':        THEME['secondary'],
    'grid.linewidth':    0.5,
    'grid.alpha':        0.7,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.titlepad':     10,
    'figure.dpi':        120,
})

custom_cmap = LinearSegmentedColormap.from_list(
    'premium', [THEME['bg'], THEME['primary'], THEME['accent'], THEME['dark']]
)
div_cmap = LinearSegmentedColormap.from_list(
    'diverging', [THEME['accent'], THEME['bg'], THEME['primary']]
)

PL = dict(
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['bg'],
    font=dict(color=THEME['dark'], family='Arial, sans-serif', size=12),
    title_font=dict(size=18, color=THEME['dark'], family='Arial, sans-serif'),
    margin=dict(l=50, r=40, t=70, b=50),
    hoverlabel=dict(bgcolor=THEME['dark'], font_color='white', font_size=12),
)

def styled_ax(ax, title='', subtitle=''):
    if title:
        ax.set_title(title, fontweight='bold', fontsize=13,
                     loc='left', color=THEME['dark'], pad=26)
    if subtitle:
        ax.text(0, 1.03, subtitle, transform=ax.transAxes,
                fontsize=9, color=THEME['accent'], style='italic',
                va='bottom', clip_on=False)
    ax.spines['left'].set_color(THEME['secondary'])
    ax.spines['bottom'].set_color(THEME['secondary'])
    return ax

def save_fig(fig, name):
    fig.savefig(f'{FIGURES_DIR}/{name}', dpi=150, bbox_inches='tight',
                facecolor=THEME['bg'])

RANDOM_STATE = 42
FIGURES_DIR  = 'reports/figures'
MODELS_DIR   = 'models'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('Theme engine initialized.')
print(f'Color system: {THEME}')

Theme engine initialized.
Color system: {'bg': '#F7F9FC', 'secondary': '#190608', 'primary': '#750D1C', 'accent': '#35369c', 'dark': '#1D1A39'}


In [ ]:
try:
    from google.colab import files
    print("Colab detected — select 'Student Depression Dataset.csv':")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
except Exception:
    DATA_PATH = '../data/Student Depression Dataset.csv'
    df_raw = pd.read_csv(DATA_PATH)

print(f'Dataset loaded | Shape: {df_raw.shape}')
df_raw.head(3)

---
## 1. Data Quality Assessment

Before any analysis, we audit the raw dataset for structural issues — missing values, duplicates, impossible entries, and near-zero variance columns.

In [ ]:
df = df_raw.copy()

# ── Missing values ─────────────────────────────
missing = df.isnull().sum()
pct     = missing / len(df) * 100
missing_report = pd.DataFrame({'Count': missing, 'Pct': pct.round(2)})
missing_report = missing_report[missing_report['Count'] > 0].sort_values('Pct', ascending=False)

if not missing_report.empty:
    display(
        missing_report.style
        .background_gradient(subset=['Pct'], cmap=custom_cmap)
        .format({'Pct': '{:.2f}%'})
        .set_caption('Missing Values per Column')
    )
else:
    print('No missing values found.')

before = len(df)
df = df.dropna(subset=['Financial Stress']).reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with null Financial Stress | Shape: {df.shape}')

# ── Duplicates ─────────────────────────────────
n_dup = df.duplicated().sum()
if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicates removed: {n_dup} | Shape: {df.shape}')

# ── CGPA = 0 ───────────────────────────────────
n_zero = (df['CGPA'] == 0).sum()
df = df[df['CGPA'] != 0].reset_index(drop=True)
print(f'CGPA=0 removed: {n_zero} | Shape: {df.shape}')

# ── Age outliers ───────────────────────────────
df = df[df['Age'] <= 60].reset_index(drop=True)
print(f'Age>60 removed | Shape: {df.shape}')

# ── Rare Others ────────────────────────────────
before = len(df)
df = df[df['Sleep Duration'] != 'Others']
df = df[df['Dietary Habits'] != 'Others']
df = df.reset_index(drop=True)
print(f"'Others' rows removed: {before - len(df)} | Shape: {df.shape}")

df_clean = df.copy()
print(f'\nClean dataset ready: {df_clean.shape}')

In [ ]:
from scipy.stats import gaussian_kde as _gkde

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(THEME['bg'])

# ── Panel 1: Missing values bar ────────────────────────────────────────────
ax = axes[0]
if not missing_report.empty:
    max_pct = missing_report['Pct'].max()
    bars = ax.barh(
        missing_report.index, missing_report['Pct'],
        color=THEME['accent'], alpha=0.85, height=0.35,
    )
    ax.set_xlim(0, max_pct * 2.4)
    for bar, val in zip(bars, missing_report['Pct']):
        ax.text(
            bar.get_width() + max_pct * 0.10,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}%', va='center', fontsize=11,
            color=THEME['dark'], fontweight='bold',
        )
    ax.set_xlabel('Missing %')
    ax.set_yticks(range(len(missing_report)))
    ax.set_yticklabels(missing_report.index, fontsize=10)
    ax.set_ylim(-0.8, len(missing_report) - 0.2)
    ax.grid(axis='x', alpha=0.4)
    ax.grid(axis='y', alpha=0)
else:
    ax.text(0.5, 0.5, 'No Missing Values', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color=THEME['primary'],
            fontweight='bold')
    ax.set_axis_off()
styled_ax(ax, 'Missing Values', 'Raw dataset null audit')

# ── Panel 2: Class balance ─────────────────────────────────────────────────
ax = axes[1]
counts = df_clean['Depression'].value_counts().sort_index()
bars = ax.bar(
    ['No Depression', 'Depression'], counts.values,
    color=PALETTE_LIST, alpha=0.9, width=0.5,
)
for bar, val in zip(bars, counts.values):
    pct = val / counts.sum() * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + counts.max() * 0.01,
        f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=11,
        color=THEME['dark'], fontweight='bold',
    )
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.18)
styled_ax(ax, 'Class Distribution', 'Target variable balance')

# ── Panel 3: Age distribution — scipy KDE (no duplicate legend labels) ────
ax = axes[2]
age_min = df_clean['Age'].min() - 1
age_max = df_clean['Age'].max() + 1
xs = np.linspace(age_min, age_max, 300)

for cls, color, label in [(0, C_NO, 'No Depression'), (1, C_YES, 'Depression')]:
    subset = df_clean[df_clean['Depression'] == cls]['Age'].dropna().values
    ax.hist(subset, bins=25, color=color, alpha=0.40, density=True, label=label)
    kde_fn = _gkde(subset, bw_method=0.35)
    ax.plot(xs, kde_fn(xs), color=color, linewidth=2.5)

ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.set_xlim(age_min, age_max)
ax.legend(frameon=False, fontsize=10)
styled_ax(ax, 'Age Distribution', 'Post-cleaning (Age ≤ 60)')

plt.suptitle('Data Quality Overview', fontsize=16, fontweight='bold',
             color=THEME['dark'])
plt.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, 'dqa_overview.png')
plt.show()

> **Insight**
> Only `Financial Stress` had missing values (~3 rows, < 0.02%). All other fields were complete.
> Age distribution is near-normal (18–30), with a small tail of mature students that was removed above 60.
> Class imbalance is moderate: approximately 58% of students are depressed — manageable with `class_weight='balanced'`.

---
## 2. Exploratory Data Analysis

We explore distributions, relationships, and patterns that inform the modeling strategy.

In [ ]:
NUMERIC_COLS = ['Age', 'CGPA', 'Academic Pressure', 'Work/Study Hours', 'Financial Stress']
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df_clean.columns]

from scipy.stats import gaussian_kde as _gkde_eda

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
fig.patch.set_facecolor(THEME['bg'])

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    col_all = df_clean[col].dropna()
    x_min, x_max = col_all.min(), col_all.max()
    x_pad = (x_max - x_min) * 0.04        # proportional padding — no negatives
    xs = np.linspace(x_min - x_pad, x_max + x_pad, 300)

    # Discrete variables (≤10 unique values: Academic Pressure, Financial Stress)
    # need a wider bandwidth to avoid spikes at each integer value
    bw = 0.65 if col_all.nunique() <= 10 else 0.35

    for cls, color, label in [(0, C_NO, 'No Depression'), (1, C_YES, 'Depression')]:
        data = df_clean[df_clean['Depression'] == cls][col].dropna().values
        ax.hist(data, bins=30, color=color, alpha=0.35, density=True, label=label)
        kde_fn = _gkde_eda(data, bw_method=bw)
        ax.plot(xs, kde_fn(xs), color=color, linewidth=2.5)

    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(frameon=False, fontsize=9)

    # Single-line title — no subtitle so nothing overlaps
    ax.set_title(col, fontweight='bold', fontsize=12, loc='left',
                 color=THEME['dark'], pad=6)
    ax.spines['left'].set_color(THEME['secondary'])
    ax.spines['bottom'].set_color(THEME['secondary'])

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions — KDE + Histogram Overlay',
             fontsize=15, fontweight='bold', color=THEME['dark'], y=1.01)
plt.tight_layout()
save_fig(fig, 'kde_distributions.png')
plt.show()

> **Insight**
> **Academic Pressure** and **Financial Stress** show the clearest separation between classes — depressed students cluster at higher values.
> **CGPA** shows the opposite: depressed students tend to have lower GPAs, suggesting academic underperformance as both a symptom and a stressor.

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_COLS), figsize=(20, 7))
fig.patch.set_facecolor(THEME['bg'])

# Legend patches (drawn once, placed on last axis)
legend_patches = [
    mpatches.Patch(facecolor=C_NO,  alpha=0.75, label='No Depression'),
    mpatches.Patch(facecolor=C_YES, alpha=0.75, label='Depression'),
]

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]

    # seaborn violinplot — bw_adjust controls smoothness:
    # 1.5 merges the integer-value bumps into one fluid shape
    sns.violinplot(
        data=df_clean,
        x='Depression', y=col,
        hue='Depression',
        palette=PALETTE,
        inner='quart',   # quartile lines inside
        cut=0,           # don't extend tails beyond data range
        bw_adjust=1.5,   # wider bandwidth → smooth, no integer spikes
        linewidth=0.8,
        legend=False,
        ax=ax,
    )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No\nDepression', 'Depression'], fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel(col if i == 0 else '', fontsize=10)
    ax.set_title(col, fontweight='bold', fontsize=11, loc='left', color=THEME['dark'], pad=6)
    ax.spines['left'].set_color(THEME['secondary'])
    ax.spines['bottom'].set_color(THEME['secondary'])

# Single shared legend on the last axis
axes[-1].legend(
    handles=legend_patches, frameon=False,
    fontsize=10, loc='upper right',
)

plt.suptitle('Violin Plots — Distribution Shape by Depression Class',
             fontsize=15, fontweight='bold', color=THEME['dark'], y=1.01)
plt.tight_layout()
save_fig(fig, 'violin_plots.png')
plt.show()

In [ ]:
# ── Sankey: Suicidal Thoughts -> Depression ────────────────────────────────
st_col  = 'Have you ever had suicidal thoughts ?'
fh_col  = 'Family History of Mental Illness'
tgt_col = 'Depression'

# Compute flows: (ST, FH) -> Depression
flows = (
    df_clean.groupby([st_col, fh_col, tgt_col])
    .size().reset_index(name='n')
)

# Node order:
# 0: ST No    1: ST Yes
# 2: FH No    3: FH Yes
# 4: No Dep   5: Dep
st_map = {'No': 0, 'Yes': 1}
fh_map = {'No': 2, 'Yes': 3}
tg_map = {0: 4, 1: 5}

src, tgt, val = [], [], []
# Layer 1: ST -> FH
for (st, fh), grp in flows.groupby([st_col, fh_col]):
    src.append(st_map[st]); tgt.append(fh_map[fh]); val.append(grp['n'].sum())
# Layer 2: FH -> Depression
for (fh, dep), grp in flows.groupby([fh_col, tgt_col]):
    src.append(fh_map[fh]); tgt.append(tg_map[dep]); val.append(grp['n'].sum())

node_colors = [
    f"rgba(185,209,117,0.85)",  # ST No
    f"rgba(163,62,121,0.85)",   # ST Yes
    f"rgba(185,209,117,0.70)",  # FH No
    f"rgba(163,62,121,0.70)",   # FH Yes
    f"rgba(185,209,117,0.95)",  # No Depression
    f"rgba(163,62,121,0.95)",   # Depression
]

fig_sankey = go.Figure(go.Sankey(
    arrangement='snap',
    textfont=dict(size=12, color=THEME['dark']),
    node=dict(
        label=['No Suicidal\nThoughts', 'Had Suicidal\nThoughts',
               'No Family\nHistory', 'Family\nHistory',
               'Not\nDepressed', 'Depressed'],
        color=node_colors,
        pad=20, thickness=22,
        line=dict(color=THEME['dark'], width=0.5),
    ),
    link=dict(
        source=src, target=tgt, value=val,
        color=[f'rgba(69,12,63,0.15)'] * len(src),
    ),
))

fig_sankey.update_layout(
    title=dict(text='Patient Flow: Suicidal Thoughts → Family History → Depression',
               font=dict(size=17, color=THEME['dark'])),
    **PL,
)
fig_sankey.show()

> **Insight**
> Students who have had suicidal thoughts are dramatically more likely to be depressed — and family history amplifies this risk further.
> The sankey reveals that the path **Suicidal Thoughts YES + Family History YES** flows almost entirely into the Depressed node.

In [ ]:
def encode_for_corr(df):
    d = df.copy()
    sleep_map = {'Less than 5 hours': 1, '5-6 hours': 2, '6-7 hours': 3,
                 '7-8 hours': 4, 'More than 8 hours': 5}
    if 'Sleep Duration' in d.columns:
        d['Sleep Duration'] = d['Sleep Duration'].map(sleep_map)
    for col in ['Have you ever had suicidal thoughts ?',
                'Family History of Mental Illness', 'Gender']:
        if col in d.columns:
            u = d[col].dropna().unique()
            if len(u) == 2:
                d[col] = d[col].map({u[0]: 0, u[1]: 1})
    obj = d.select_dtypes('object').columns.tolist()
    if obj:
        d = pd.get_dummies(d, columns=obj, drop_first=True)
    return d

df_enc  = encode_for_corr(df_clean)
df_num  = df_enc.select_dtypes(include=[np.number])
corr    = df_num.corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor(THEME['bg'])

sns.heatmap(
    corr, mask=mask, cmap=div_cmap, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 8, 'color': THEME['dark']},
    linewidths=0.4, linecolor=THEME['secondary'],
    square=True, ax=ax, cbar_kws={'shrink': 0.7},
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
styled_ax(ax, 'Correlation Heatmap', 'Pearson r — lower triangle only')
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'corr_heatmap.png')
plt.show()

print('Top 8 correlations with Depression:')
print(corr['Depression'].abs().sort_values(ascending=False).head(9).iloc[1:].round(3).to_string())

In [ ]:
corrs = {}
for col in df_num.columns:
    if col == 'Depression':
        continue
    v = df_num[[col, 'Depression']].dropna()
    if v[col].std() == 0:
        continue
    r, _ = pointbiserialr(v['Depression'], v[col])
    corrs[col] = r

pb = pd.Series(corrs).abs().sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(THEME['bg'])

colors = [THEME['accent'] if v > pb.median() else THEME['primary'] for v in pb.values]
bars = ax.barh(pb.index, pb.values, color=colors, alpha=0.88, height=0.65)
for bar, val in zip(bars, pb.values):
    ax.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9.5, color=THEME['dark'])

ax.axvline(pb.median(), color=THEME['dark'], linestyle='--',
           linewidth=1, alpha=0.5, label='Median')
ax.set_xlabel('|Point-Biserial r|')
ax.legend(frameon=False, fontsize=9)
styled_ax(ax, 'Feature Predictive Power',
          'Point-biserial correlation with Depression target — top 12')
ax.grid(axis='x', alpha=0.4)
ax.grid(axis='y', alpha=0)

plt.tight_layout()
save_fig(fig, 'pb_correlation.png')
plt.show()

top5 = pb.sort_values(ascending=False).head(5).index.tolist()
top5_raw = [f for f in top5 if f in df_clean.columns]
print(f'Top 5 predictive features: {top5_raw}')

> **Insight**
> `Have you ever had suicidal thoughts?` is the single strongest predictor — highlighting the direct psychological dimension of depression.
> `Academic Pressure`, `Financial Stress`, and `Sleep Duration` form the next tier, confirming that environmental and behavioral stressors are the core drivers.

In [ ]:
CAT_COLS = ['Gender', 'Sleep Duration', 'Dietary Habits', 'Degree',
            'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']
CAT_COLS = [c for c in CAT_COLS if c in df_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
fig.patch.set_facecolor(THEME['bg'])

for i, col in enumerate(CAT_COLS):
    ax = axes[i]
    dep_rate = df_clean.groupby(col)['Depression'].mean().sort_values(ascending=True)
    counts   = df_clean[col].value_counts()
    dep_rate = dep_rate.reindex(dep_rate.index)

    colors = [THEME['accent'] if v > dep_rate.median() else THEME['primary']
              for v in dep_rate.values]
    bars = ax.barh(dep_rate.index.astype(str), dep_rate.values,
                   color=colors, alpha=0.85, height=0.6)
    ax.axvline(dep_rate.median(), color=THEME['dark'], linestyle='--',
               linewidth=1, alpha=0.5)
    for bar, val in zip(bars, dep_rate.values):
        ax.text(bar.get_width() + 0.008, bar.get_y() + bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=9, color=THEME['dark'])
    ax.set_xlabel('Depression Rate')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.grid(axis='y', alpha=0)
    ax.grid(axis='x', alpha=0.4)
    styled_ax(ax, col, 'Depression rate per category')

for j in range(len(CAT_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Depression Rate by Categorical Features',
             fontsize=15, fontweight='bold', color=THEME['dark'])
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, 'categorical_depression_rate.png')
plt.show()

In [ ]:
counts = df_clean['Depression'].value_counts().sort_index()

fig_donut = go.Figure(go.Pie(
    labels=['No Depression', 'Depression'],
    values=counts.values,
    hole=0.55,
    marker=dict(
        colors=[THEME['primary'], THEME['accent']],
        line=dict(color=THEME['bg'], width=3),
    ),
    textinfo='label+percent',
    textfont=dict(size=14, color=THEME['dark']),
    hovertemplate='%{label}<br>Count: %{value:,}<br>Share: %{percent}<extra></extra>',
))

fig_donut.add_annotation(
    text=f'<b>{counts.sum():,}</b><br>students',
    x=0.5, y=0.5, showarrow=False,
    font=dict(size=16, color=THEME['dark']),
    align='center',
)

fig_donut.update_layout(
    title=dict(text='Target Class Distribution', font=dict(size=18)),
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15),
    **PL,
)
fig_donut.show()

---
## 3. Feature Engineering

We transform raw columns into model-ready features:
1. Drop irrelevant columns (`id`, `City`, `Work Pressure`, `Job Satisfaction`, `Profession`)
2. Create interaction feature: `Academic Pressure × Sleep Duration`
3. Binary-encode Yes/No columns
4. Ordinal-encode Sleep Duration (1–5 scale)
5. One-hot encode Gender, Dietary Habits, Degree
6. MinMax scale all numeric features
7. 80/20 stratified split
8. RFE feature selection (top 15)

In [ ]:
DROP_COLS   = ['id', 'Profession', 'Work Pressure', 'Job Satisfaction', 'City']
SLEEP_ORDER = ['Less than 5 hours', '5-6 hours', '6-7 hours', '7-8 hours', 'More than 8 hours']
SLEEP_MAP   = {v: i+1 for i, v in enumerate(SLEEP_ORDER)}
OHE_COLS    = ['Gender', 'Dietary Habits', 'Degree']
BINARY_MAP  = {
    'Have you ever had suicidal thoughts ?': {'Yes': 1, 'No': 0},
    'Family History of Mental Illness':      {'Yes': 1, 'No': 0},
}
FEAT_NUM    = ['Age', 'CGPA', 'Academic Pressure', 'Work/Study Hours', 'Financial Stress']
N_RFE       = 15

def prepare_features(df):
    df = df.copy()
    y  = df['Depression'].values
    df = df.drop(columns=['Depression'])

    to_drop = [c for c in DROP_COLS if c in df.columns]
    if 'City' in df.columns and df['City'].nunique() > 20 and 'City' not in to_drop:
        to_drop.append('City')
    df.drop(columns=to_drop, errors='ignore', inplace=True)

    sl_enc = df['Sleep Duration'].map(SLEEP_MAP).fillna(3) if 'Sleep Duration' in df.columns else 3
    df['AP_x_Sleep'] = df['Academic Pressure'] * sl_enc

    for col, mapping in BINARY_MAP.items():
        if col in df.columns:
            df[col] = df[col].map(mapping)

    if 'Sleep Duration' in df.columns:
        df['Sleep Duration'] = df['Sleep Duration'].map(SLEEP_MAP).fillna(3)

    num_feats  = [c for c in FEAT_NUM + ['AP_x_Sleep'] if c in df.columns]
    ohe_feats  = [c for c in OHE_COLS if c in df.columns]
    pass_feats = [c for c in df.columns if c not in num_feats and c not in ohe_feats]

    transformers = [
        ('num', MinMaxScaler(), num_feats),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_feats),
    ]
    if pass_feats:
        transformers.append(('pass', 'passthrough', pass_feats))
    prep = ColumnTransformer(transformers)

    Xtr_raw, Xte_raw, ytr, yte = train_test_split(
        df, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    Xtr = prep.fit_transform(Xtr_raw)
    Xte = prep.transform(Xte_raw)

    ohe_names = (
        prep.named_transformers_['ohe'].get_feature_names_out(ohe_feats).tolist()
        if ohe_feats else []
    )
    feat_names = num_feats + ohe_names + pass_feats

    rfe = RFE(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
        n_features_to_select=min(N_RFE, Xtr.shape[1]), step=1
    )
    rfe.fit(Xtr, ytr)

    sel = [f for f, s in zip(feat_names, rfe.support_) if s]
    dropped = [f for f, s in zip(feat_names, rfe.support_) if not s]

    print(f'Selected ({len(sel)}): {sel}')
    print(f'Dropped  ({len(dropped)}): {dropped}')

    return rfe.transform(Xtr), rfe.transform(Xte), ytr, yte, sel, prep, rfe, Xte_raw


X_train, X_test, y_train, y_test, selected_features, preprocessor, rfe, X_test_raw = prepare_features(df_clean)
print(f'\nTrain: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(THEME['bg'])

n_total = len(selected_features) + len(
    [f for f, s in zip(selected_features, rfe.support_) if not s]
)
bars = ax.barh(range(len(selected_features)), [1]*len(selected_features),
               color=THEME['primary'], alpha=0.8, height=0.6)
for j, feat in enumerate(selected_features):
    ax.text(0.02, j, feat, va='center', ha='left', fontsize=10, color=THEME['dark'])
ax.set_xlim(0, 1.4)
ax.set_yticks([])
ax.set_xticks([])
ax.spines['bottom'].set_visible(False)
styled_ax(ax, f'RFE Selected Features ({len(selected_features)} of all)',
          'Recursive Feature Elimination with Logistic Regression estimator')

plt.tight_layout()
save_fig(fig, 'rfe_features.png')
plt.show()

---
## 4. Model Training

Five classifiers trained with **Stratified 5-Fold Cross-Validation**. Class imbalance handled with `class_weight='balanced'`. Best model then fine-tuned with `GridSearchCV`.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    'f1_macro':        'f1_macro',
    'roc_auc':         'roc_auc',
    'accuracy':        'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro':    'recall_macro',
}

def cv_scores(model, X, y):
    res = cross_validate(model, X, y, cv=CV, scoring=SCORING, n_jobs=-1)
    return {m + s: float(np.mean(res[f'test_{m}']) if s=='_mean'
                         else np.std(res[f'test_{m}']))
            for m in SCORING for s in ('_mean', '_std')}

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
spw   = n_neg / n_pos if n_pos > 0 else 1.0

model_specs = [
    ('Logistic Regression',
     LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
     {'class_weight': 'balanced'}),
    ('Decision Tree',
     DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=RANDOM_STATE),
     {'max_depth': 5}),
    ('Random Forest',
     RandomForestClassifier(class_weight='balanced', n_estimators=200,
                            random_state=RANDOM_STATE, n_jobs=-1),
     {'n_estimators': 200}),
]
if HAS_XGB:
    model_specs.append((
        'XGBoost',
        XGBClassifier(scale_pos_weight=spw, n_estimators=200,
                      random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
        {'scale_pos_weight': round(spw, 3)},
    ))
if HAS_LGB:
    model_specs.append((
        'LightGBM',
        LGBMClassifier(is_unbalance=True, n_estimators=200,
                       random_state=RANDOM_STATE, verbose=-1),
        {'is_unbalance': True},
    ))

records = []
for name, model, _ in model_specs:
    print(f'Training {name} ...', end=' ')
    t0 = time.time()
    sc = cv_scores(model, X_train, y_train)
    elapsed = time.time() - t0
    records.append({'model': name, 'time_s': round(elapsed, 1), **sc})
    print(f'F1={sc["f1_macro_mean"]:.4f} | AUC={sc["roc_auc_mean"]:.4f} | {elapsed:.1f}s')

results_df = pd.DataFrame(records).sort_values('f1_macro_mean', ascending=False).reset_index(drop=True)
print('\nBase training complete.')

In [ ]:
best_base = results_df.iloc[0]['model']
print(f'Best base model: {best_base} — running GridSearchCV...')

if 'Random Forest' in best_base:
    base_est = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    pgrid = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
elif 'XGBoost' in best_base and HAS_XGB:
    base_est = XGBClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE,
                             eval_metric='logloss', verbosity=0)
    pgrid = {'learning_rate': [0.05, 0.1], 'max_depth': [4, 6],
             'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}
elif 'LightGBM' in best_base and HAS_LGB:
    base_est = LGBMClassifier(is_unbalance=True, random_state=RANDOM_STATE, verbose=-1)
    pgrid = {'learning_rate': [0.05, 0.1], 'max_depth': [4, 6],
             'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}
else:
    base_est = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    pgrid = {'C': [0.01, 0.1, 1.0, 10.0]}

grid = GridSearchCV(base_est, pgrid, cv=CV, scoring='f1_macro', n_jobs=-1, refit=True)
t0   = time.time()
grid.fit(X_train, y_train)
elap = time.time() - t0

best_model = grid.best_estimator_
print(f'Best params: {grid.best_params_}')
print(f'GridSearch CV F1: {grid.best_score_:.4f} | Time: {elap:.1f}s')

tuned_sc = cv_scores(best_model, X_train, y_train)
records.append({'model': best_base + ' (Tuned)', 'time_s': round(elap, 1), **tuned_sc})

best_model.fit(X_train, y_train)
with open(f'{MODELS_DIR}/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Model saved.')

results_df = pd.DataFrame(records).sort_values('f1_macro_mean', ascending=False).reset_index(drop=True)

In [ ]:
train_sizes, train_sc, val_sc = learning_curve(
    best_model, X_train, y_train,
    cv=CV, scoring='f1_macro', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
)

tr_mean, tr_std = train_sc.mean(axis=1), train_sc.std(axis=1)
vl_mean, vl_std = val_sc.mean(axis=1), val_sc.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(THEME['bg'])

ax.plot(train_sizes, tr_mean, color=THEME['primary'], lw=2.5, label='Training score')
ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std,
                alpha=0.15, color=THEME['primary'])
ax.plot(train_sizes, vl_mean, color=THEME['accent'], lw=2.5, label='CV score')
ax.fill_between(train_sizes, vl_mean - vl_std, vl_mean + vl_std,
                alpha=0.15, color=THEME['accent'])

gap = tr_mean[-1] - vl_mean[-1]
ax.annotate(f'Gap: {gap:.3f}',
            xy=(train_sizes[-1], (tr_mean[-1] + vl_mean[-1]) / 2),
            xytext=(train_sizes[-2] - 1500, (tr_mean[-1] + vl_mean[-1]) / 2 + 0.01),
            arrowprops=dict(arrowstyle='->', color=THEME['dark'], lw=1.2),
            fontsize=10, color=THEME['dark'])

ax.set_xlabel('Training Set Size')
ax.set_ylabel('F1-macro Score')
ax.legend(frameon=False, fontsize=11)
ax.set_ylim(0.7, 1.02)
styled_ax(ax, 'Learning Curve', f'Best model: {best_base} — bias/variance diagnosis')

plt.tight_layout()
save_fig(fig, 'learning_curve.png')
plt.show()

---
## 5. Model Comparison

All models evaluated on the same 5-fold CV. Tuned model highlighted in the comparison.

In [ ]:
metrics_radar = ['f1_macro_mean', 'roc_auc_mean', 'accuracy_mean',
                  'precision_macro_mean', 'recall_macro_mean']
metric_labels = ['F1-Macro', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall']

colors_radar = [
    THEME['primary'], '#7B9E3A', '#C8A2C8',
    THEME['accent'], '#E07B54', '#5BA4CF',
]

fig_radar = go.Figure()
for idx, row in results_df.iterrows():
    vals = [row[m] for m in metrics_radar]
    vals_closed = vals + [vals[0]]
    labels_closed = metric_labels + [metric_labels[0]]
    color = colors_radar[idx % len(colors_radar)]
    is_best = '(Tuned)' in row['model']
    r_hex = color.lstrip('#')
    r_rgb = tuple(int(r_hex[i:i+2], 16) for i in (0, 2, 4))
    fill_color = f'rgba({r_rgb[0]},{r_rgb[1]},{r_rgb[2]},0.20)' if is_best else 'rgba(0,0,0,0)'
    fig_radar.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=labels_closed,
        fill='toself',
        name=row['model'],
        line=dict(color=color, width=3 if is_best else 1.5),
        opacity=1.0,
        fillcolor=fill_color,
    ))

fig_radar.update_layout(
    polar=dict(
        bgcolor='white',
        radialaxis=dict(
            visible=True, range=[0.75, 1.0],
            tickfont=dict(size=10, color=THEME['dark']),
            gridcolor=THEME['secondary'], linecolor=THEME['secondary'],
        ),
        angularaxis=dict(
            tickfont=dict(size=12, color=THEME['dark']),
            linecolor=THEME['secondary'],
        ),
    ),
    title=dict(text='Model Performance — Radar Comparison', font=dict(size=18)),
    legend=dict(orientation='v', x=1.05, y=0.5,
                font=dict(size=11, color=THEME['dark'])),
    **PL,
)
fig_radar.show()

In [ ]:
show_cols = ['model', 'f1_macro_mean', 'f1_macro_std',
             'roc_auc_mean', 'accuracy_mean', 'time_s']
rename_map = {
    'model': 'Model', 'f1_macro_mean': 'F1-macro',
    'f1_macro_std': 'F1 Std', 'roc_auc_mean': 'ROC-AUC',
    'accuracy_mean': 'Accuracy', 'time_s': 'Time (s)',
}

disp = results_df[show_cols].rename(columns=rename_map).set_index('Model')

best_idx = disp['F1-macro'].idxmax()

def highlight_best(row):
    return ['background-color: ' + THEME['secondary'] + '; font-weight: bold'
            if row.name == best_idx else '' for _ in row]

styled = (
    disp.style
    .apply(highlight_best, axis=1)
    .background_gradient(subset=['F1-macro', 'ROC-AUC', 'Accuracy'],
                         cmap=custom_cmap, vmin=0.75, vmax=1.0)
    .bar(subset=['Time (s)'], color=THEME['secondary'], vmin=0)
    .format({'F1-macro': '{:.4f}', 'F1 Std': '{:.4f}',
             'ROC-AUC': '{:.4f}', 'Accuracy': '{:.4f}', 'Time (s)': '{:.1f}'})
    .set_caption('Cross-Validation Results — Sorted by F1-macro')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '14px'), ('font-weight', 'bold'),
                  ('color', THEME['dark']), ('text-align', 'left')]
    }])
)
display(styled)

---
## 6. Model Evaluation

Held-out test set (20%) evaluation of the best tuned model.

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = (
    best_model.predict_proba(X_test)[:, 1]
    if hasattr(best_model, 'predict_proba')
    else best_model.decision_function(X_test)
)

report = classification_report(y_test, y_pred,
    target_names=['No Depression', 'Depression'])
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor(THEME['bg'])

sns.heatmap(
    cm, annot=False, cmap=custom_cmap,
    xticklabels=['No Depression', 'Depression'],
    yticklabels=['No Depression', 'Depression'],
    linewidths=2, linecolor=THEME['bg'],
    ax=ax, cbar_kws={'shrink': 0.7},
)
for _i in range(2):
    for _j in range(2):
        _norm = cm[_i, _j] / cm.max()
        _tc = 'white' if _norm > 0.55 else THEME['dark']
        ax.text(_j + 0.5, _i + 0.5, str(cm[_i, _j]),
                ha='center', va='center',
                fontsize=22, fontweight='bold', color=_tc)

ax.set_xlabel('Predicted', fontsize=12, labelpad=10)
ax.set_ylabel('Actual', fontsize=12, labelpad=10)
ax.xaxis.set_label_position('bottom')
ax.xaxis.tick_bottom()
styled_ax(ax, 'Confusion Matrix', f'Test set  |  n = {len(y_test):,}')
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

tn, fp, fn, tp = cm.ravel()
ax.annotate(f'Sensitivity (Recall): {tp/(tp+fn):.3f}',
            xy=(1.02, 0.15), xycoords='axes fraction',
            fontsize=10, color=THEME['dark'])
ax.annotate(f'Specificity: {tn/(tn+fp):.3f}',
            xy=(1.02, 0.05), xycoords='axes fraction',
            fontsize=10, color=THEME['dark'])

plt.tight_layout()
save_fig(fig, 'confusion_matrix.png')
plt.show()

In [ ]:
fpr, tpr, _     = roc_curve(y_test, y_prob)
auc_score       = roc_auc_score(y_test, y_prob)
prec, rec, _    = precision_recall_curve(y_test, y_prob)
ap_score        = average_precision_score(y_test, y_prob)

fig_curves = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'ROC Curve  (AUC = {auc_score:.4f})',
        f'Precision-Recall  (AP = {ap_score:.4f})',
    ],
)

# ROC
fig_curves.add_trace(
    go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC',
               line=dict(color=THEME['accent'], width=3)),
    row=1, col=1,
)
fig_curves.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random',
               line=dict(color=THEME['primary'], width=1.5, dash='dash')),
    row=1, col=1,
)

# PR
fig_curves.add_trace(
    go.Scatter(x=rec, y=prec, mode='lines', name='PR Curve',
               line=dict(color=THEME['dark'], width=3)),
    row=1, col=2,
)
baseline = y_test.mean()
fig_curves.add_trace(
    go.Scatter(x=[0, 1], y=[baseline, baseline], mode='lines',
               name=f'Baseline ({baseline:.2f})',
               line=dict(color=THEME['primary'], width=1.5, dash='dash')),
    row=1, col=2,
)

fig_curves.update_xaxes(title_text='False Positive Rate', row=1, col=1,
                         gridcolor=THEME['secondary'])
fig_curves.update_yaxes(title_text='True Positive Rate',  row=1, col=1,
                         gridcolor=THEME['secondary'])
fig_curves.update_xaxes(title_text='Recall', row=1, col=2,
                         gridcolor=THEME['secondary'])
fig_curves.update_yaxes(title_text='Precision', row=1, col=2,
                         gridcolor=THEME['secondary'])

fig_curves.update_layout(
    title=dict(text='Model Evaluation Curves', font=dict(size=18)),
    height=480,
    **PL,
)
fig_curves.show()

In [ ]:
fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor(THEME['bg'])

ax.plot([0, 1], [0, 1], linestyle='--', color=THEME['primary'],
        linewidth=2, label='Perfectly calibrated')
ax.plot(mean_pred, fraction_pos, 'o-', color=THEME['accent'],
        linewidth=2.5, markersize=8, label=best_base + ' (Tuned)', zorder=5)

for x, y in zip(mean_pred, fraction_pos):
    ax.annotate(f'{y:.2f}', (x, y), textcoords='offset points',
                xytext=(6, 4), fontsize=8.5, color=THEME['dark'])

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.legend(frameon=False, fontsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.10)
styled_ax(ax, 'Calibration Curve',
          'How reliable are the predicted probabilities?')

plt.tight_layout()
save_fig(fig, 'calibration_curve.png')
plt.show()

> **Insight**
> The calibration curve shows how closely the model's predicted probabilities match true outcomes.
> Points near the diagonal mean the model is well-calibrated — a probability of 0.8 really does mean ~80% of those students are depressed.
> Significant deviation would indicate the model is over- or under-confident.

---
## 7. Model Explainability — SHAP

SHAP (SHapley Additive exPlanations) shows **which features push individual predictions** up or down — and by how much.

In [ ]:
if HAS_SHAP:
    try:
        if hasattr(best_model, 'feature_importances_'):
            explainer   = shap.TreeExplainer(best_model)
            shap_values = explainer.shap_values(X_test)
            if isinstance(shap_values, list):
                sv = shap_values[1]
            else:
                sv = shap_values
        else:
            explainer   = shap.LinearExplainer(best_model, X_train)
            shap_values = explainer.shap_values(X_test)
            sv = shap_values

        # ── Summary dot plot ─────────────────────────────────
        fig_shap, ax_shap = plt.subplots(figsize=(10, 7))
        fig_shap.patch.set_facecolor(THEME['bg'])
        ax_shap.set_facecolor(THEME['bg'])
        shap.summary_plot(
            sv, X_test,
            feature_names=selected_features,
            plot_type='dot',
            color_bar=True,
            show=False,
            plot_size=None,
        )
        plt.title('SHAP Summary — Feature Impact on Depression Prediction',
                  fontweight='bold', fontsize=13, loc='left', color=THEME['dark'], pad=10)
        plt.tight_layout()
        save_fig(plt.gcf(), 'shap_summary.png')
        plt.show()

        # ── Bar chart of mean |SHAP| ─────────────────────────
        mean_shap = pd.Series(np.abs(sv).mean(axis=0),
                              index=selected_features).sort_values()
        fig2, ax2 = plt.subplots(figsize=(9, 6))
        fig2.patch.set_facecolor(THEME['bg'])
        colors_shap = [THEME['accent'] if v > mean_shap.median()
                       else THEME['primary'] for v in mean_shap.values]
        ax2.barh(mean_shap.index, mean_shap.values,
                 color=colors_shap, alpha=0.88, height=0.65)
        for i, (feat, val) in enumerate(mean_shap.items()):
            ax2.text(val + 0.001, i, f'{val:.4f}', va='center',
                     fontsize=9, color=THEME['dark'])
        styled_ax(ax2, 'Mean |SHAP| Value per Feature',
                  'Average absolute impact on model output')
        ax2.set_xlabel('Mean |SHAP value|')
        ax2.grid(axis='x', alpha=0.4)
        ax2.grid(axis='y', alpha=0)
        plt.tight_layout()
        save_fig(fig2, 'shap_bar.png')
        plt.show()

        top3 = mean_shap.sort_values(ascending=False).head(3)
        print('\nTop 3 drivers of Depression:')
        for rank, (feat, val) in enumerate(top3.items(), 1):
            print(f'  {rank}. {feat}  (mean |SHAP| = {val:.4f})')

    except Exception as e:
        print(f'SHAP error: {e}')
else:
    print('Install shap: !pip install shap')

---
### SHAP Interpretation

The SHAP summary plot reveals three consistent patterns:

1. **Suicidal Thoughts** — the single largest driver. A `Yes` answer strongly shifts the prediction toward Depression.
2. **Academic Pressure** — high values push predictions toward Depression across most students.
3. **Sleep Duration** — insufficient sleep (< 6 hrs) is a strong positive SHAP contributor.

Notably, **Financial Stress** and **CGPA** interact: students with both high financial stress *and* low CGPA show compounding risk that the model has learned to detect.

> **Key takeaway:** Depression in this dataset is not driven by a single factor — it emerges from the compound effect of psychological signals (suicidal thoughts), environmental stressors (academic pressure, finances), and behavioral indicators (sleep, diet).

---
## 8. Error Analysis

We examine the cases the model gets wrong — False Positives and False Negatives — to understand where predictions fail and why.

In [ ]:
y_true_arr = np.array(y_test)
y_pred_arr = np.array(y_pred)

fp_mask = (y_pred_arr == 1) & (y_true_arr == 0)
fn_mask = (y_pred_arr == 0) & (y_true_arr == 1)
tp_mask = (y_pred_arr == 1) & (y_true_arr == 1)
tn_mask = (y_pred_arr == 0) & (y_true_arr == 0)

X_raw_reset = X_test_raw.reset_index(drop=True)
fp_df = X_raw_reset[fp_mask].copy()
fn_df = X_raw_reset[fn_mask].copy()
tp_df = X_raw_reset[tp_mask].copy()
tn_df = X_raw_reset[tn_mask].copy()

print(f'True Positives : {tp_mask.sum():5d} ({tp_mask.mean():.1%})')
print(f'True Negatives : {tn_mask.sum():5d} ({tn_mask.mean():.1%})')
print(f'False Positives: {fp_mask.sum():5d} ({fp_mask.mean():.1%})')
print(f'False Negatives: {fn_mask.sum():5d} ({fn_mask.mean():.1%})')

numeric_err = ['Age', 'CGPA', 'Academic Pressure', 'Work/Study Hours', 'Financial Stress']
numeric_err = [c for c in numeric_err if c in X_raw_reset.columns]

profile = pd.DataFrame({
    'True Positive':  tp_df[numeric_err].mean().round(2) if len(tp_df) > 0 else pd.Series(),
    'True Negative':  tn_df[numeric_err].mean().round(2) if len(tn_df) > 0 else pd.Series(),
    'False Positive': fp_df[numeric_err].mean().round(2) if len(fp_df) > 0 else pd.Series(),
    'False Negative': fn_df[numeric_err].mean().round(2) if len(fn_df) > 0 else pd.Series(),
})

display(
    profile.style
    .background_gradient(cmap=custom_cmap, axis=1)
    .set_caption('Mean Feature Values by Prediction Outcome')
)

In [ ]:
fig, axes = plt.subplots(1, len(numeric_err), figsize=(20, 6))
fig.patch.set_facecolor(THEME['bg'])

labels = ['FP', 'FN', 'TP', 'TN']
colors = [THEME['accent'], THEME['dark'], THEME['primary'], THEME['secondary']]
dfs    = [fp_df, fn_df, tp_df, tn_df]

for i, col in enumerate(numeric_err):
    ax = axes[i]
    data = [d[col].dropna().values for d in dfs]
    vp = ax.violinplot(data, positions=range(len(labels)),
                       showmeans=True, widths=0.7)
    for j, pc in enumerate(vp['bodies']):
        pc.set_facecolor(colors[j])
        pc.set_alpha(0.7)
    for part in ['cmeans', 'cbars', 'cmins', 'cmaxes']:
        if part in vp:
            vp[part].set_color(THEME['dark'])
            vp[part].set_linewidth(1.2)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title(col, fontweight='bold', fontsize=10, loc='left', color=THEME['dark'])
    ax.spines['left'].set_color(THEME['secondary'])
    ax.spines['bottom'].set_color(THEME['secondary'])

plt.suptitle('Feature Distribution by Prediction Outcome  (FP | FN | TP | TN)',
             fontsize=14, fontweight='bold', color=THEME['dark'], y=1.02)
plt.tight_layout()
save_fig(fig, 'error_analysis.png')
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score

thresholds = np.linspace(0.05, 0.95, 80)
thresh_records = []
for t in thresholds:
    preds = (y_prob >= t).astype(int)
    thresh_records.append({
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall':    recall_score(y_test, preds, zero_division=0),
        'f1':        f1_score(y_test, preds, zero_division=0),
    })
th_df = pd.DataFrame(thresh_records)

fig_th = go.Figure()
fig_th.add_trace(go.Scatter(
    x=th_df['threshold'], y=th_df['precision'],
    mode='lines', name='Precision',
    line=dict(color=THEME['primary'], width=2.5),
))
fig_th.add_trace(go.Scatter(
    x=th_df['threshold'], y=th_df['recall'],
    mode='lines', name='Recall',
    line=dict(color=THEME['accent'], width=2.5),
))
fig_th.add_trace(go.Scatter(
    x=th_df['threshold'], y=th_df['f1'],
    mode='lines', name='F1',
    line=dict(color=THEME['dark'], width=3, dash='dot'),
))

best_t = th_df.loc[th_df['f1'].idxmax(), 'threshold']
best_f = th_df['f1'].max()
fig_th.add_vline(
    x=best_t, line_width=1.5, line_dash='dash', line_color=THEME['dark'],
    annotation_text=f'Best F1 threshold = {best_t:.2f}',
    annotation_position='top right',
    annotation_font=dict(size=12, color=THEME['dark']),
)

fig_th.update_layout(
    title=dict(text='Precision / Recall / F1 vs Decision Threshold',
               font=dict(size=17)),
    xaxis=dict(title='Threshold', gridcolor=THEME['secondary']),
    yaxis=dict(title='Score', gridcolor=THEME['secondary']),
    legend=dict(orientation='h', y=-0.2),
    height=460,
    **PL,
)
fig_th.show()
print(f'Optimal threshold (max F1): {best_t:.2f} -> F1 = {best_f:.4f}')

---
## 9. Final Summary

In [ ]:
test_f1  = f1_score(y_test, y_pred, average='macro')
best_row = results_df.iloc[0]

# ── Metric cards (Plotly) ────────────────────────────────────────────────────
metrics = [
    ('Test F1-macro',  f'{test_f1:.4f}'),
    ('Test ROC-AUC',   f'{auc_score:.4f}'),
    ('Test Avg Prec.', f'{ap_score:.4f}'),
    ('CV F1-macro',    f"{best_row['f1_macro_mean']:.4f} ± {best_row['f1_macro_std']:.4f}"),
    ('Best Model',     best_row['model']),
    ('Features Used',  str(len(selected_features))),
]

fig_cards = make_subplots(
    rows=1,
    cols=len(metrics),
    subplot_titles=[m[0] for m in metrics],
    specs=[[{'type': 'indicator'} for _ in metrics]],
)

for col_idx, (label, value) in enumerate(metrics, 1):
    color = THEME['accent'] if col_idx <= 3 else THEME['primary']
    display_value = str(value)
    font_size = 22

    if label == 'CV F1-macro':
        display_value = display_value.replace(' ± ', '<br>± ')
        font_size = 16
    elif label == 'Best Model' and len(display_value) > 18:
        words = display_value.split(' ')
        split_at = max(1, len(words) // 2)
        display_value = ' '.join(words[:split_at]) + '<br>' + ' '.join(words[split_at:])
        font_size = 16

    fig_cards.add_trace(
        go.Indicator(
            mode='number',
            value=None,
            title=dict(text=display_value,
                       font=dict(size=font_size, color=color)),
            number=dict(font=dict(size=1, color=THEME['bg'])),
        ),
        row=1, col=col_idx,
    )

fig_cards.update_layout(
    **{**PL,
       'height': 220,
       'title': dict(text='Final Results Dashboard', font=dict(size=18)),
       'margin': dict(t=80, b=20, l=20, r=20)},
)
fig_cards.show()

# ── Text summary ─────────────────────────────────────────────────────────────
print('=' * 60)
print('STUDENT DEPRESSION PREDICTION — FINAL RESULTS')
print('=' * 60)
print(f'Clean dataset        : {df_clean.shape[0]:,} students, {df_clean.shape[1]} columns')
print(f'Best model           : {best_row["model"]}')
print(f'CV F1-macro          : {best_row["f1_macro_mean"]:.4f} +/- {best_row["f1_macro_std"]:.4f}')
print(f'Test F1-macro        : {test_f1:.4f}')
print(f'Test ROC-AUC         : {auc_score:.4f}')
print(f'Avg Precision        : {ap_score:.4f}')
print(f'Features selected    : {len(selected_features)}')
print(f'  Top features       : {selected_features[:5]}')
print(f'Optimal threshold    : {best_t:.2f} (max F1 = {best_f:.4f})')
print('=' * 60)
print(f'Figures saved to     : {FIGURES_DIR}/')
print(f'Model saved to       : {MODELS_DIR}/best_model.pkl')

---
## 8. Fairness, Significance, and Robustness Add-ons

This section extends the baseline notebook to satisfy checklist-oriented advanced analysis requirements.

**Decision Note:** We keep Logistic Regression as primary model, but add complementary methods (PCA, Chi-Square, resampling, progressive sampling, subgroup fairness, significance checks) to demonstrate methodological depth and critical evaluation.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report
from sklearn.base import clone
from sklearn.pipeline import Pipeline

# 1) PCA diagnostic (variance explanation)
pca_probe = PCA().fit(X_train)
cum_var = np.cumsum(pca_probe.explained_variance_ratio_)
min_components_95 = int(np.argmax(cum_var >= 0.95) + 1)

# 2) Chi-square feature ranking (requires non-negative inputs)
chi_selector = SelectKBest(score_func=chi2, k=min(10, X_train.shape[1]))
chi_selector.fit(X_train, y_train)
chi_idx = chi_selector.get_support(indices=True)
chi_feats = [selected_features[i] for i in chi_idx if i < len(selected_features)]

# 3) Progressive sampling sanity-check
sample_fracs = [0.2, 0.4, 0.6, 0.8, 1.0]
prog_records = []
for frac in sample_fracs:
    n = int(len(X_train) * frac)
    clf = clone(best_model)
    clf.fit(X_train[:n], y_train[:n])
    pred = clf.predict(X_test)
    prog_records.append({
        'train_fraction': frac,
        'train_rows': n,
        'f1_macro': f1_score(y_test, pred, average='macro')
    })
progressive_df = pd.DataFrame(prog_records)

print(f'PCA components for >=95% variance: {min_components_95}')
print('Chi-Square top features:', chi_feats)
display(progressive_df.style.format({'f1_macro': '{:.4f}'}).set_caption('Progressive Sampling Check'))

In [ ]:
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

resampling_methods = {
    'Baseline (no resampling)': None,
    'RandomUnderSampler': RandomUnderSampler(random_state=RANDOM_STATE),
    'RandomOverSampler': RandomOverSampler(random_state=RANDOM_STATE),
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
}

resampling_records = []
for method_name, sampler in resampling_methods.items():
    if sampler is None:
        Xr, yr = X_train, y_train
    else:
        Xr, yr = sampler.fit_resample(X_train, y_train)

    clf = clone(best_model)
    clf.fit(Xr, yr)
    pred = clf.predict(X_test)
    resampling_records.append({
        'method': method_name,
        'train_size_after_sampling': len(yr),
        'test_f1_macro': f1_score(y_test, pred, average='macro'),
        'test_recall_depression': recall_score(y_test, pred, pos_label=1),
    })

resampling_df = pd.DataFrame(resampling_records).sort_values('test_f1_macro', ascending=False)
display(resampling_df.style.format({'test_f1_macro': '{:.4f}', 'test_recall_depression': '{:.4f}'}).set_caption('Imbalanced Data Strategy Comparison'))

print('Decision Note: We keep class_weight-based LR as the primary approach unless resampling gives stable gains.')

In [36]:
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.base import clone

# -------------------------
# Safety defaults
# -------------------------
if 'RANDOM_STATE' not in globals():
    RANDOM_STATE = 42

# -------------------------
# Local fallback prepare_features
# -------------------------
def prepare_features_local(df):
    DROP_COLS = ['id', 'Profession', 'Work Pressure', 'Job Satisfaction', 'City']
    SLEEP_ORDER = ['Less than 5 hours', '5-6 hours', '6-7 hours', '7-8 hours', 'More than 8 hours']
    SLEEP_MAP = {v: i + 1 for i, v in enumerate(SLEEP_ORDER)}
    OHE_COLS = ['Gender', 'Dietary Habits', 'Degree']
    BINARY_MAP = {
        'Have you ever had suicidal thoughts ?': {'Yes': 1, 'No': 0},
        'Family History of Mental Illness': {'Yes': 1, 'No': 0},
    }
    FEAT_NUM = ['Age', 'CGPA', 'Academic Pressure', 'Work/Study Hours', 'Financial Stress']
    N_RFE = 15

    d = df.copy()
    y = d['Depression'].values
    d = d.drop(columns=['Depression'])

    to_drop = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=to_drop, errors='ignore', inplace=True)

    sl_enc = d['Sleep Duration'].map(SLEEP_MAP).fillna(3) if 'Sleep Duration' in d.columns else 3
    d['AP_x_Sleep'] = d['Academic Pressure'] * sl_enc

    for col, mapping in BINARY_MAP.items():
        if col in d.columns:
            d[col] = d[col].map(mapping)

    if 'Sleep Duration' in d.columns:
        d['Sleep Duration'] = d['Sleep Duration'].map(SLEEP_MAP).fillna(3)

    num_feats = [c for c in FEAT_NUM + ['AP_x_Sleep'] if c in d.columns]
    ohe_feats = [c for c in OHE_COLS if c in d.columns]
    pass_feats = [c for c in d.columns if c not in num_feats and c not in ohe_feats]

    transformers = [
        ('num', MinMaxScaler(), num_feats),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_feats),
    ]
    if pass_feats:
        transformers.append(('pass', 'passthrough', pass_feats))

    prep = ColumnTransformer(transformers)

    Xtr_raw, Xte_raw, ytr, yte = train_test_split(
        d, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    Xtr = prep.fit_transform(Xtr_raw)
    Xte = prep.transform(Xte_raw)

    ohe_names = (
        prep.named_transformers_['ohe'].get_feature_names_out(ohe_feats).tolist()
        if ohe_feats else []
    )
    feat_names = num_feats + ohe_names + pass_feats

    rfe = RFE(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
        n_features_to_select=min(N_RFE, Xtr.shape[1]),
        step=1
    )
    rfe.fit(Xtr, ytr)

    sel = [f for f, s in zip(feat_names, rfe.support_) if s]
    return rfe.transform(Xtr), rfe.transform(Xte), ytr, yte, sel, prep, rfe, Xte_raw

# -------------------------
# Build missing dependencies
# -------------------------
if 'df_clean' not in globals():
    raise ValueError("df_clean tanımlı değil. Önce veri yükleme + DQA hücrelerini çalıştır.")

if 'X_test_raw' not in globals() or 'X_train' not in globals():
    X_train, X_test, y_train, y_test, selected_features, preprocessor, rfe, X_test_raw = prepare_features_local(df_clean)

if 'best_model' not in globals():
    best_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    best_model.fit(X_train, y_train)

if 'y_pred' not in globals():
    y_pred = best_model.predict(X_test)

# -------------------------
# Speed config
# -------------------------
FAST_MODE = True
FAST_FRACTION = 0.5
FAST_FOLDS = 3
FULL_FOLDS = 5

# -------------------------
# Fairness quick check
# -------------------------
fairness_rows = []
X_eval = X_test_raw.reset_index(drop=True).copy()

def subgroup_metrics(mask, subgroup_name):
    if mask.sum() < 30:
        return
    yt = y_test[mask]
    yp = y_pred[mask]
    fairness_rows.append({
        'subgroup': subgroup_name,
        'n': int(mask.sum()),
        'f1_macro': f1_score(yt, yp, average='macro'),
        'recall_depression': recall_score(yt, yp, pos_label=1, zero_division=0),
    })

if 'Gender' in X_eval.columns:
    for g in sorted(X_eval['Gender'].dropna().unique()):
        subgroup_metrics((X_eval['Gender'] == g).values, f'Gender={g}')

if 'Academic Pressure' in X_eval.columns:
    subgroup_metrics((X_eval['Academic Pressure'] <= 2).values, 'AcademicPressure<=2')
    subgroup_metrics((X_eval['Academic Pressure'] >= 4).values, 'AcademicPressure>=4')

fairness_df = pd.DataFrame(fairness_rows)
if not fairness_df.empty:
    display(fairness_df.style.format({'f1_macro': '{:.4f}', 'recall_depression': '{:.4f}'}).set_caption('Subgroup Fairness Snapshot'))

# -------------------------
# Significance: LR vs SVM
# -------------------------
try:
    n_train = len(X_train)
    if FAST_MODE:
        n_use = max(5000, int(n_train * FAST_FRACTION))
        X_sig = X_train[:n_use]
        y_sig = y_train[:n_use]
        cv_sig = StratifiedKFold(n_splits=FAST_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        print(f'FAST_MODE ON -> using {n_use:,}/{n_train:,} rows, {FAST_FOLDS}-fold CV')
    else:
        X_sig = X_train
        y_sig = y_train
        cv_sig = StratifiedKFold(n_splits=FULL_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        print(f'FAST_MODE OFF -> full train, {FULL_FOLDS}-fold CV')

    svm = SVC(C=1.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=False, random_state=RANDOM_STATE)
    lr_for_cv = clone(best_model)

    cv_lr = cross_validate(lr_for_cv, X_sig, y_sig, cv=cv_sig, scoring='f1_macro', n_jobs=-1)['test_score']
    cv_svm = cross_validate(svm, X_sig, y_sig, cv=cv_sig, scoring='f1_macro', n_jobs=-1)['test_score']
    _, p_val = ttest_rel(cv_lr, cv_svm)

    svm.fit(X_sig, y_sig)
    y_pred_svm = svm.predict(X_test)

    b = int(((y_pred == y_test) & (y_pred_svm != y_test)).sum())
    c = int(((y_pred != y_test) & (y_pred_svm == y_test)).sum())
    mc = mcnemar([[0, b], [c, 0]], exact=False, correction=True)

    print(f'Paired t-test p-value (LR vs SVM): {p_val:.4f}')
    print(f'McNemar p-value (LR vs SVM): {mc.pvalue:.4f}')
except Exception as e:
    print('Significance block skipped:', e)

,subgroup,n,f1_macro,recall_depression
0,Gender=Female,2413,0.8319,0.8463
1,Gender=Male,3159,0.8380,0.8307
2,AcademicPressure<=2,1857,0.7737,0.5856
3,AcademicPressure>=4,2235,0.7679,0.9252


FAST_MODE ON -> using 11,143/22,287 rows, 3-fold CV
Paired t-test p-value (LR vs SVM): 0.4582
McNemar p-value (LR vs SVM): 0.1594


---
## 9. Model Card Summary, Ethics, and Limitations

### Intended Use
- Educational data analysis project for SENG 352.
- Early-warning style risk discussion support.

### Not Intended Use
- Clinical diagnosis or treatment decisions.
- Automated student labeling without expert oversight.

### Ethical Notes
- False negatives are critical in this domain.
- Subgroup gaps (especially low-pressure groups) must be monitored.
- Any operational use requires governance, privacy controls, and human review.

### Decision Note
The selected model is kept simple (tuned Logistic Regression) to maximize transparency and reproducibility.

---
## 10. Gradio Interactive Demo

This demo makes the model behavior easier to communicate in presentation.

### How to run in Colab
1. Run all required upstream cells (feature engineering, training, and evaluation) so `preprocessor`, `rfe`, and `best_model` are available.
2. Run the Gradio code cell below to create the `demo` object.
3. Launch the app with:
```python
demo.launch(share=True)
```
4. Open the generated public URL and test scenarios interactively.
5. Stop the app when done (interrupt cell execution).

> Clinical disclaimer: this interface is educational and should not be used as a diagnostic tool.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
from pathlib import Path

SLEEP_MAP_DEMO = {
    'Less than 5 hours': 1,
    '5-6 hours': 2,
    '6-7 hours': 3,
    '7-8 hours': 4,
    'More than 8 hours': 5,
}

DEFAULT_PROFILE = {
    'Age': 21,
    'CGPA': 7.0,
    'Sleep Duration': 3,
    'Academic Pressure': 3,
    'Work/Study Hours': 6,
    'Financial Stress': 3,
    'Study Satisfaction': 3,
    'Have you ever had suicidal thoughts ?': 0,
    'Family History of Mental Illness': 0,
    'Gender': 'Male',
    'Dietary Habits': 'Moderate',
    'Degree': 'BSc',
    'AP_x_Sleep': 9,
}


def _build_single_row(age, cgpa, sleep_duration, academic_pressure, work_study_hours, financial_stress,
                      study_satisfaction, suicidal, family_history, gender, dietary, degree):
    sleep_num = SLEEP_MAP_DEMO.get(sleep_duration, 3)
    row = DEFAULT_PROFILE.copy()
    row.update({
        'Age': age,
        'CGPA': cgpa,
        'Sleep Duration': sleep_num,
        'Academic Pressure': academic_pressure,
        'Work/Study Hours': work_study_hours,
        'Financial Stress': financial_stress,
        'Study Satisfaction': study_satisfaction,
        'Have you ever had suicidal thoughts ?': 1 if suicidal == 'Yes' else 0,
        'Family History of Mental Illness': 1 if family_history == 'Yes' else 0,
        'Gender': gender,
        'Dietary Habits': dietary,
        'Degree': degree,
        'AP_x_Sleep': academic_pressure * sleep_num,
    })

    df_one = pd.DataFrame([row])
    X_one = preprocessor.transform(df_one)
    X_one = rfe.transform(X_one)
    return X_one


def _risk_band(prob):
    if prob >= 0.75:
        return 'High risk', 'High confidence high-risk signal.'
    if prob >= 0.50:
        return 'Elevated risk', 'Borderline high-risk signal.'
    if prob >= 0.25:
        return 'Moderate-low risk', 'Borderline low-risk signal.'
    return 'Low risk', 'High confidence low-risk signal.'


def predict_risk_demo(age, cgpa, sleep_duration, academic_pressure, work_study_hours, financial_stress,
                      study_satisfaction, suicidal, family_history, gender, dietary, degree):
    try:
        X_one = _build_single_row(age, cgpa, sleep_duration, academic_pressure, work_study_hours, financial_stress,
                                  study_satisfaction, suicidal, family_history, gender, dietary, degree)

        proba = float(best_model.predict_proba(X_one)[0, 1])
        band, risk_text = _risk_band(proba)
        label = 'High Risk' if proba >= 0.5 else 'Low Risk'

        report_md = (
            f"### Prediction Summary\n"
            f"- **Predicted class:** `{label}`\n"
            f"- **Depression probability:** `{proba:.3f}`\n"
            f"- **Risk band:** `{band}`\n\n"
            f"**Interpretation:** {risk_text}\n\n"
            f"> Educational use only. Not a clinical diagnostic output."
        )
        return label, proba, band, report_md
    except Exception as e:
        return 'Error', 0.0, 'N/A', f'### Error\nInput/processing error: {e}'


def preset_low_risk():
    return [20, 8.2, '7-8 hours', 2, 5, 2, 4, 'No', 'No', 'Female', 'Healthy', 'BSc']


def preset_medium_risk():
    return [22, 6.8, '6-7 hours', 3, 8, 3, 3, 'No', 'Yes', 'Male', 'Moderate', 'BSc']


def preset_high_risk():
    return [24, 5.4, 'Less than 5 hours', 5, 12, 5, 1, 'Yes', 'Yes', 'Male', 'Unhealthy', 'BSc']


custom_css = """
#hero-card {
    border: 1px solid #d7dbe3;
    border-radius: 14px;
    padding: 14px 18px;
    background: linear-gradient(135deg, #f8fafc 0%, #eef2ff 100%);
}
#kpi-box {
    border: 1px solid #d7dbe3;
    border-radius: 12px;
    padding: 8px;
}
"""

with gr.Blocks(
    title='Student Depression Risk Demo',
    theme=gr.themes.Soft(primary_hue='indigo', neutral_hue='slate'),
    css=custom_css,
) as demo:
    # Prefer local/Colab-friendly image names first, then fallback paths
    hero_candidates = [
        Path('hero_image.png'),
        Path('hero_image.jpg'),
        Path('/content/hero_image.png'),
        Path('/content/hero_image.jpg'),
        Path('C:/Users/ASUS/.cursor/projects/c-Users-ASUS-OneDrive-Desktop-student-depression-prediction/assets/c__Users_ASUS_AppData_Roaming_Cursor_User_workspaceStorage_2dfd883f7638e874253ce035ad9e2cfb_images_image-6bfd3a9c-577a-46ea-8769-6b4b92c32fd0.png'),
    ]

    hero_image = None
    for p in hero_candidates:
        if p.exists():
            hero_image = np.array(Image.open(p).convert('RGB'))
            break

    with gr.Row(elem_id='hero-card'):
        with gr.Column(scale=2):
            gr.Markdown(
                """
                # Student Depression Risk Analyzer
                Professional demo interface for the trained ML pipeline.

                - Uses your real preprocessor + feature selector + trained model
                - Provides probability, class, and risk-band explanation
                - Educational decision-support only (not a clinical diagnosis)
                """
            )
        with gr.Column(scale=1):
            if hero_image is not None:
                gr.Image(
                    value=hero_image,
                    show_label=False,
                    show_download_button=False,
                    interactive=False,
                    height=260,
                    elem_id='kpi-box',
                )
            else:
                gr.Markdown('**Tip:** upload `hero_image.png` to show a custom hero visual.')

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown('### Input Profile')
            age = gr.Slider(15, 40, value=21, step=1, label='Age')
            cgpa = gr.Slider(0.0, 10.0, value=7.0, step=0.1, label='CGPA')
            sleep = gr.Dropdown(
                ['Less than 5 hours', '5-6 hours', '6-7 hours', '7-8 hours', 'More than 8 hours'],
                value='6-7 hours',
                label='Sleep Duration'
            )
            ap = gr.Slider(1, 5, value=3, step=1, label='Academic Pressure')
            wsh = gr.Slider(0, 16, value=6, step=1, label='Work/Study Hours')
            fs = gr.Slider(1, 5, value=3, step=1, label='Financial Stress')
            ss = gr.Slider(1, 5, value=3, step=1, label='Study Satisfaction')
            su = gr.Dropdown(['Yes', 'No'], value='No', label='Suicidal Thoughts')
            fh = gr.Dropdown(['Yes', 'No'], value='No', label='Family History of Mental Illness')
            gen = gr.Dropdown(['Male', 'Female'], value='Male', label='Gender')
            diet = gr.Dropdown(['Healthy', 'Moderate', 'Unhealthy'], value='Moderate', label='Dietary Habits')
            deg = gr.Textbox(value='BSc', label='Degree')

            with gr.Row():
                low_btn = gr.Button('Load Low-Risk Example')
                med_btn = gr.Button('Load Mid-Risk Example')
                high_btn = gr.Button('Load High-Risk Example')

            with gr.Row():
                run_btn = gr.Button('Analyze Profile', variant='primary')
                clear_btn = gr.ClearButton(
                    components=[age, cgpa, sleep, ap, wsh, fs, ss, su, fh, gen, diet, deg],
                    value='Clear Inputs'
                )

        with gr.Column(scale=1):
            gr.Markdown('### Model Output')
            out_label = gr.Textbox(label='Predicted Class', elem_id='kpi-box')
            out_prob = gr.Number(label='Depression Probability', precision=3, elem_id='kpi-box')
            out_band = gr.Textbox(label='Risk Band', elem_id='kpi-box')
            out_text = gr.Markdown(label='Interpretation')

    inputs = [age, cgpa, sleep, ap, wsh, fs, ss, su, fh, gen, diet, deg]
    outputs = [out_label, out_prob, out_band, out_text]

    run_btn.click(fn=predict_risk_demo, inputs=inputs, outputs=outputs)
    low_btn.click(fn=preset_low_risk, inputs=None, outputs=inputs)
    med_btn.click(fn=preset_medium_risk, inputs=None, outputs=inputs)
    high_btn.click(fn=preset_high_risk, inputs=None, outputs=inputs)

# For Colab usage:
# demo.launch(share=True)
print('Gradio demo object is ready. Run demo.launch(share=True) in Colab to open UI.')

---
## 10b. Additional Checklist Coverage (Aggregation + Standardization + SVD)

This optional block closes checklist gaps with lightweight supplementary analyses.

**Decision Note:** These are supporting experiments for methodological completeness; the primary production pipeline remains the same.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD

supp_results = {}

# 1) Aggregation example
agg_cols = [c for c in ['Academic Pressure', 'Financial Stress', 'CGPA', 'Work/Study Hours'] if c in df_clean.columns]
if agg_cols and 'Gender' in df_clean.columns:
    agg_df = (
        df_clean.groupby(['Gender', 'Depression'])[agg_cols]
        .mean()
        .round(2)
        .reset_index()
    )
    display(agg_df.style.set_caption('Aggregation Example: Mean Features by Gender x Depression'))

# 2) StandardScaler supplementary comparison (vs current MinMax setup)
try:
    num_idx = list(range(min(6, X_train.shape[1])))
    scaler_std = StandardScaler()
    X_train_std = X_train.copy()
    X_test_std = X_test.copy()
    X_train_std[:, num_idx] = scaler_std.fit_transform(X_train[:, num_idx])
    X_test_std[:, num_idx] = scaler_std.transform(X_test[:, num_idx])

    lr_std = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    lr_std.fit(X_train_std, y_train)
    y_pred_std = lr_std.predict(X_test_std)
    f1_std = f1_score(y_test, y_pred_std, average='macro')
    f1_minmax = f1_score(y_test, y_pred, average='macro')

    supp_results['MinMax (Main Pipeline)'] = f1_minmax
    supp_results['StandardScaler (Supplementary)'] = f1_std

    print(f'StandardScaler supplementary F1-macro: {f1_std:.4f}')
    print(f'Current MinMax pipeline F1-macro    : {f1_minmax:.4f}')
except Exception as e:
    print('StandardScaler supplementary block skipped:', e)

# 3) SVD supplementary check
try:
    n_comp = min(10, X_train.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    X_train_svd = svd.fit_transform(X_train)
    X_test_svd = svd.transform(X_test)

    lr_svd = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    lr_svd.fit(X_train_svd, y_train)
    y_pred_svd = lr_svd.predict(X_test_svd)
    f1_svd = f1_score(y_test, y_pred_svd, average='macro')

    supp_results[f'SVD ({n_comp} comps) + LR'] = f1_svd

    print(f'TruncatedSVD ({n_comp} comps) + LR F1-macro: {f1_svd:.4f}')
    print(f'Explained variance ratio sum: {svd.explained_variance_ratio_.sum():.4f}')
except Exception as e:
    print('SVD supplementary block skipped:', e)

# 4) Visual summary chart
if supp_results:
    supp_df = pd.DataFrame({'Approach': list(supp_results.keys()), 'F1_macro': list(supp_results.values())})
    supp_df = supp_df.sort_values('F1_macro', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    fig.patch.set_facecolor(THEME['bg'])

    colors = [THEME['accent'] if 'Main Pipeline' in a else THEME['primary'] for a in supp_df['Approach']]
    bars = ax.barh(supp_df['Approach'], supp_df['F1_macro'], color=colors, alpha=0.85)

    for b in bars:
        ax.text(b.get_width() + 0.0005, b.get_y() + b.get_height()/2, f"{b.get_width():.4f}", va='center', fontsize=10)

    ax.set_xlabel('F1-macro')
    ax.set_xlim(max(0.75, supp_df['F1_macro'].min() - 0.01), min(1.0, supp_df['F1_macro'].max() + 0.01))
    styled_ax(ax, 'Supplementary Scaling/Reduction Comparison', 'Checklist closure experiment (higher is better)')
    ax.grid(axis='x', alpha=0.35)
    ax.grid(axis='y', alpha=0)

    plt.tight_layout()
    plt.show()

---
## 11. Final Results and Checklist Evidence Matrix

This final block maps project outputs to checklist expectations and demonstrates traceability.

**Decision Note:** Methods not used in the core pipeline are either added as supplementary experiments or explicitly justified.

In [ ]:
evidence_rows = [
    ('Missing values', 'Used', 'DQA block: Financial Stress null handling'),
    ('Outlier handling', 'Used', 'DQA block: CGPA=0 and age filtering decisions'),
    ('One-Hot Encoding', 'Used', 'Feature engineering with ColumnTransformer'),
    ('Min-max normalization', 'Used', 'Main pipeline uses MinMaxScaler'),
    ('Standardization', 'Supplementary', 'Section 10b compares StandardScaler performance'),
    ('Attribute Selection', 'Used', 'RFE-based feature selection'),
    ('Aggregation', 'Supplementary', 'Section 10b groupby summary (Gender x Depression)'),
    ('RFE', 'Used', 'Feature selection from 43 to 15 features'),
    ('PCA', 'Supplementary', 'Section 8 variance diagnostic'),
    ('SVD', 'Supplementary', 'Section 10b TruncatedSVD + LR check'),
    ('Chi-Square feature selection', 'Supplementary', 'Section 8 SelectKBest chi2'),
    ('Stratified sampling', 'Used', 'train_test_split(..., stratify=y)'),
    ('Progressive sampling', 'Supplementary', 'Section 8 fractional training check'),
    ('K-fold Cross Validation', 'Used', 'Stratified CV in model comparison'),
    ('Balanced class weight', 'Used', "class_weight='balanced' in classifiers"),
    ('Under/Over sampling + SMOTE', 'Supplementary', 'Section 8 resampling comparison'),
    ('Systematic/Cluster/Weighted sampling', 'N/A (justified)', 'Not core for this tabular supervised setup; stratified strategy used instead'),
    ('Text preprocessing items', 'N/A (justified)', 'Dataset is structured tabular data, not raw text corpus'),
    ('Representation learning (autoencoder etc.)', 'N/A (justified)', 'Out of scope for baseline tabular pipeline; explainability/fairness prioritized'),
    ('UMAP/t-SNE', 'Used in project modules', 'Implemented in src/embedding_viz.py and analysis log'),
    ('Gradio demo', 'Used', 'Section 10 interactive interface for presentation'),
]

evidence_df = pd.DataFrame(evidence_rows, columns=['Checklist Item', 'Status', 'Evidence'])
display(evidence_df.style.set_caption('SENG 352 Checklist Evidence Matrix'))